<a href="https://colab.research.google.com/github/lyj3222/Lazada-/blob/main/lazada_dashboard.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## STEP 1. 환경 준비

In [9]:
# 1-1. 라이브러리 설치 및 환경 설정
!pip install openai gradio plotly -q
!apt-get install fonts-nanum -y -q

import os
import re
import json
import sqlite3
import warnings
import pandas as pd
import numpy as np
import matplotlib
import matplotlib.pyplot as plt
import matplotlib.font_manager as fm
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from openai import OpenAI
import gradio as gr

warnings.filterwarnings('ignore')

# 한글 폰트 설정
font_path = '/usr/share/fonts/truetype/nanum/NanumGothic.ttf'
if os.path.exists(font_path):
    font_prop = fm.FontProperties(fname=font_path)
    matplotlib.rcParams['font.family'] = font_prop.get_name()
    plt.rcParams['axes.unicode_minus'] = False

try:
    from google.colab import userdata
    os.environ['OPENAI_API_KEY'] = userdata.get('OPENAI_API_KEY')
except:
    pass

client = OpenAI(api_key=os.environ.get('OPENAI_API_KEY'))

# 환율 상수
IDR_TO_USD = 1 / 16200
USD_TO_KRW = 1380
IDR_TO_KRW = IDR_TO_USD * USD_TO_KRW

def idr_to_usd(val): return round(val * IDR_TO_USD, 2)
def idr_to_krw(val): return round(val * IDR_TO_KRW)

print('✅ 환경 준비 완료 (환율 함수 포함)')

Reading package lists...
Building dependency tree...
Reading state information...
fonts-nanum is already the newest version (20200506-1).
0 upgraded, 0 newly installed, 0 to remove and 3 not upgraded.
✅ 환경 준비 완료 (환율 함수 포함)


## STEP 2. 데이터 탐색 (EDA)

In [10]:
import pandas as pd

CSV_PATH = '전처리_중_라자다_EN.csv'

try:
    df_raw = pd.read_csv(CSV_PATH)
    print(f'✅ 파일 로드 성공: {CSV_PATH}')
    print(f'행: {df_raw.shape[0]:,}, 열: {df_raw.shape[1]}')
    display(df_raw.head(3))
except FileNotFoundError:
    print(f"❌ 파일을 찾을 수 없습니다: {CSV_PATH}")

✅ 파일 로드 성공: 전처리_중_라자다_EN.csv
행: 591, 열: 29


,product_url,title,rating,reviews,initial_price,final_price,currency,images,seller_name,breadcrumb,...,color,return_warranty,is_super_seller,promotions,brand,selected_options,is_lazmall,domain,number_sold,gmv
0,https://www.lazada.co.id/products/dioda-damper...,DIODA DAMPER DMV 1500 TV POLYTRON NEW ORIGINAL...,4.6,27,0,10000,IDR,"[""https://img.lazcdn.com/g/ff/kf/Sc92f4b3c99c0...",YUDIANA YUDI SPERPAT ELEKTRONIK,"[""Televisi & Video"",""Televisi Digital""]",...,없음,"[""Berubah Pikiran"",""7 Hari Gratis Pengembalian...",FALSE,[],TR,[],FALSE,https://www.lazada.co.id,111,1110000.0
1,https://www.lazada.co.id/products/victus-lapto...,Victus Laptop Gaming HP AMD Ryzen 5 NVIDIA GeF...,0.0,0,17555000,16499000,IDR,"[""https://img.lazcdn.com/g/p/3440632c069eddd82...",HP,"[""Komputer & Laptop"",""Laptop"",""Laptop Umum""]",...,없음,"[""Produk Original"",""Berubah Pikiran"",""30 Hari ...",TRUE,[],HP,"[{""name"":""Varian"",""value"":""15-fb2666AX""}]",TRUE,https://www.lazada.co.id,0,0.0
2,https://www.lazada.co.id/products/laptop-hp-15...,Laptop HP 15s-fq5148TU Core i3 UHD 4GB & 8GB R...,5.0,47,6499000,6099000,IDR,"[""https://img.lazcdn.com/g/p/23782d5f1bd37a89c...",HP,"[""Komputer & Laptop"",""Laptop"",""Laptop Umum""]",...,14s-dq5115TU,"[""Produk Original"",""Berubah Pikiran"",""30 Hari ...",TRUE,[],HP,"[{""name"":""Warna"",""value"":""14s-dq5115TU""}]",TRUE,https://www.lazada.co.id,91,555009000.0


## STEP 3. 데이터 정제

In [11]:
import pandas as pd
import numpy as np
import json

col_map = {
    '상품 URL':      'product_url',
    '상품명':        'title',
    '평점':          'rating',
    '리뷰 수':       'reviews',
    '정가':          'initial_price',
    '최종 판매가':   'final_price',
    '통화':          'currency',
    '이미지':        'images',
    '판매자명':      'seller_name',
    '카테고리 경로': 'breadcrumb',
    '상품 사양':     'specs',
    '상품 설명':     'description',
    '판매자 평점':   'seller_rating',
    '정시 배송률':   'ship_on_time',
    '채팅 응답률':   'chat_response',
    'SKU':           'sku',
    'MPN':           'mpn',
    '색상(목록)':    'color_list',
    '옵션/변형':     'variations',
    '색상':          'color',
    '반품 및 보증':  'return_warranty',
    '슈퍼셀러 여부': 'is_super_seller',
    '프로모션':      'promotions',
    '브랜드':        'brand_name',
    '선택된 옵션':   'selected_options',
    '라즈몰 여부':   'is_lazmall',
    '도메인':        'domain',
    '판매 수량':     'number_sold',
    '총 거래액(GMV)':'gmv',
}

try:
    df = df_raw.rename(columns=col_map).copy()

    if 'brand' in df.columns and 'brand_name' not in df.columns:
        df = df.rename(columns={'brand': 'brand_name'})

    if 'final_price' in df.columns:
        df = df[df['final_price'] >= 100]

    df['rating'] = pd.to_numeric(df['rating'], errors='coerce').replace(0, np.nan)
    df['number_sold'] = pd.to_numeric(df['number_sold'], errors='coerce').fillna(0).astype(int)

    for col in ['is_super_seller', 'is_lazmall']:
        if col in df.columns:
            df[col] = df[col].astype(str).str.upper().map({'TRUE': 1, 'FALSE': 0, '1': 1, '0': 0, '1.0': 1, '0.0': 0})

    df['final_price_usd'] = df['final_price'].apply(idr_to_usd)
    df['final_price_krw'] = df['final_price'].apply(idr_to_krw)
    df['gmv_usd'] = df['gmv'].apply(idr_to_usd)
    df['gmv_krw'] = df['gmv'].apply(idr_to_krw)

    def parse_bc(s):
        try:
            a = json.loads(str(s).replace("'", '"'))
            return pd.Series([a[0] if len(a)>0 else '미분류', a[1] if len(a)>1 else '미분류', a[2] if len(a)>2 else '미분류'])
        except: return pd.Series(['미분류','미분류','미분류'])

    if 'breadcrumb' in df.columns:
        df[['level1','level2','level3']] = df['breadcrumb'].apply(parse_bc)
    else:
        df[['level1','level2','level3']] = '미분류'

    df['brand_name'] = df.get('brand_name', pd.Series(dtype='object')).fillna('None')
    print(f'✅ 데이터 정제 완료: {len(df)}건')
except Exception as e:
    print(f'❌ 정제 중 오류 발생: {e}')

✅ 데이터 정제 완료: 591건


## STEP 4. DB 생성 및 적재

In [12]:
import sqlite3

try:
    conn = sqlite3.connect('lazada.db', check_same_thread=False)
    cur = conn.cursor()

    cur.executescript('''
    DROP TABLE IF EXISTS products; DROP TABLE IF EXISTS sellers; DROP TABLE IF EXISTS brands; DROP TABLE IF EXISTS categories;
    CREATE TABLE sellers (id INTEGER PRIMARY KEY, name TEXT UNIQUE, rating REAL, ship_on_time REAL, chat_response REAL, is_super_seller INTEGER, is_lazmall INTEGER);
    CREATE TABLE brands (id INTEGER PRIMARY KEY, name TEXT UNIQUE);
    CREATE TABLE categories (id INTEGER PRIMARY KEY, level1 TEXT, level2 TEXT, level3 TEXT);
    CREATE TABLE products (id INTEGER PRIMARY KEY, sku TEXT UNIQUE, title TEXT, initial_price REAL, final_price REAL, final_price_usd REAL, final_price_krw INTEGER, rating REAL, reviews INTEGER, number_sold INTEGER, gmv REAL, gmv_usd REAL, gmv_krw INTEGER, seller_id INTEGER, brand_id INTEGER, category_id INTEGER);
    ''')

    sellers_data = df[['seller_name','seller_rating','ship_on_time','chat_response','is_super_seller','is_lazmall']].drop_duplicates('seller_name')
    sellers_data = sellers_data.rename(columns={'seller_name':'name','seller_rating':'rating'})
    sellers_data.to_sql('sellers', conn, if_exists='append', index=False)

    brands_data = df[['brand_name']].drop_duplicates().rename(columns={'brand_name':'name'})
    brands_data.to_sql('brands', conn, if_exists='append', index=False)

    cats_data = df[['level1','level2','level3']].drop_duplicates()
    cats_data.to_sql('categories', conn, if_exists='append', index=False)

    s_map = pd.read_sql('SELECT id as seller_id, name as seller_name FROM sellers', conn)
    b_map = pd.read_sql('SELECT id as brand_id, name as brand_name FROM brands', conn)
    c_map = pd.read_sql('SELECT id as category_id, level1, level2, level3 FROM categories', conn)

    df_p = df.merge(s_map, on='seller_name').merge(b_map, on='brand_name').merge(c_map, on=['level1','level2','level3'])
    target_cols = ['sku','title','initial_price','final_price','final_price_usd','final_price_krw','rating','reviews','number_sold','gmv','gmv_usd','gmv_krw','seller_id','brand_id','category_id']
    df_p[target_cols].to_sql('products', conn, if_exists='append', index=False)

    print('✅ DB 데이터 적재 완료 (수정된 스키마 반영)')
except Exception as e:
    print(f'❌ DB 적재 중 오류 발생: {e}')

✅ DB 데이터 적재 완료 (수정된 스키마 반영)


In [13]:
import sqlite3
import pandas as pd

# 데이터베이스 스키마 정보 조회
def get_schema(conn):
    cur = conn.cursor()
    cur.execute("SELECT name FROM sqlite_master WHERE type='table';")
    tables = cur.fetchall()

    for table_name in tables:
        table_name = table_name[0]
        print(f'\n[Table: {table_name}]')
        display(pd.read_sql(f'PRAGMA table_info({table_name});', conn)[['name', 'type']])

get_schema(conn)


[Table: sellers]


,name,type
0,id,INTEGER
1,name,TEXT
2,rating,REAL
3,ship_on_time,REAL
4,chat_response,REAL
5,is_super_seller,INTEGER
6,is_lazmall,INTEGER



[Table: brands]


,name,type
0,id,INTEGER
1,name,TEXT



[Table: categories]


,name,type
0,id,INTEGER
1,level1,TEXT
2,level2,TEXT
3,level3,TEXT



[Table: products]


,name,type
0,id,INTEGER
1,sku,TEXT
2,title,TEXT
3,initial_price,REAL
4,final_price,REAL
5,final_price_usd,REAL
6,final_price_krw,INTEGER
7,rating,REAL
8,reviews,INTEGER
9,number_sold,INTEGER


## STEP 5. 대시보드 및 AI 로직 정의

In [14]:
import plotly.express as px
import plotly.graph_objects as go

def chart_category_gmv():
    # 카테고리별 총 매출(GMV_KRW)
    cat_gmv = df_p.groupby('level1')['gmv_krw'].sum().reset_index().sort_values('gmv_krw', ascending=False).head(10)
    fig = px.bar(cat_gmv, x='level1', y='gmv_krw', title='카테고리별 총 매출 (Top 10)', labels={'level1': '카테고리', 'gmv_krw': '매출액(원)'})
    return fig

def chart_seller_compare():
    # 판매자별 평균 평점 및 응답률 비교
    seller_stats = df_p.groupby('seller_name').agg({'rating': 'mean', 'number_sold': 'sum'}).reset_index().sort_values('number_sold', ascending=False).head(10)
    fig = px.scatter(seller_stats, x='rating', y='number_sold', text='seller_name', size='number_sold', title='판매자 평점 vs 판매량 (Top 10 Sellers)')
    return fig

def chart_rating_hist():
    # 평점 분포 히스토그램
    fig = px.histogram(df_p, x='rating', nbins=20, title='상품 평점 분포', labels={'rating': '평점', 'count': '상품 수'})
    return fig

def chart_price_band():
    # 가격대별 상품 수 분포
    fig = px.box(df_p, x='level1', y='final_price_krw', title='카테고리별 가격대 분포', labels={'level1': '카테고리', 'final_price_krw': '가격(원)'})
    return fig

def chart_bestseller():
    # 베스트셀러 데이터프레임
    bestseller = df_p[['title', 'brand_name', 'number_sold', 'final_price_krw']].sort_values('number_sold', ascending=False).head(10)
    bestseller.columns = ['상품명', '브랜드', '판매량', '가격(원)']
    return bestseller

print('✅ 시각화 함수 정의 완료')

✅ 시각화 함수 정의 완료


위의 시각화 함수를 정의했으므로, 이제 아래의 Gradio 실행 셀을 다시 실행하면 대시보드에 그래프가 정상적으로 표시됩니다.

In [17]:
# ── AI 챗봇 설정 및 로직 ──────────────────────────────────────

# AI가 참조할 데이터베이스 스키마 및 규칙 정의
SYSTEM_PROMPT = """
당신은 Lazada 데이터 분석 전문가입니다. 사용자의 질문에 대해 SQLite SQL로 답하세요.

[스키마 정보]
1. products: 상품 정보
   - sku, title (상품명)
   - final_price (IDR 가격), final_price_usd (USD 가격), final_price_krw (KRW 가격)
   - rating (평점), reviews (리뷰 수), number_sold (판매량)
   - gmv_usd (총 거래액 USD), gmv_krw (총 거래액 KRW)
   - seller_id, brand_id, category_id (FK)
2. sellers: 판매자 정보
   - id, name, rating, ship_on_time (정시 배송률), chat_response (응답률), is_super_seller, is_lazmall
3. brands: 브랜드 정보 (id, name)
4. categories: 카테고리 정보 (id, level1, level2, level3)

[보안 및 중요 규칙]
- 데이터 수정(UPDATE), 삭제(DELETE), 테이블 제거(DROP), 삽입(INSERT), 구조 변경(ALTER)은 어떠한 경우에도 금지됩니다.
- 사용자가 데이터 수정을 요청할 경우, 쿼리를 생성하지 말고 즉시 거절 메시지를 유도하세요.
- SELECT 쿼리 작성을 우선으로 합니다.
- 마크다운 없이 SQL 코드만 출력하세요.
- 결과가 많을 경우 LIMIT 10을 사용하세요.
"""

def chatbot_response(user_msg, history):
    try:
        # 1. OpenAI API를 이용한 SQL 생성
        response = client.chat.completions.create(
            model='gpt-4o-mini',
            messages=[{'role':'system','content':SYSTEM_PROMPT},{'role':'user','content':user_msg}]
        )
        sql = response.choices[0].message.content.strip().replace('```sql', '').replace('```', '')

        # 2. 쿼리 실행 및 예외 처리 (수정/삭제 시도 차단 로직 강화)
        forbidden_keywords = ['UPDATE', 'DELETE', 'DROP', 'INSERT', 'ALTER', 'REPLACE', 'TRUNCATE']
        sql_upper = sql.upper()

        # SELECT로 시작하지 않거나 금지어가 포함된 경우 즉시 거절
        if not sql_upper.startswith('SELECT') or any(k in sql_upper for k in forbidden_keywords):
            return history + [(user_msg, "찾을 수 없거나 수정할 수 없습니다.")], "", None, sql

        res_df = pd.read_sql(sql, conn)

        if res_df is None or res_df.empty:
            return history + [(user_msg, "찾을 수 없거나 수정할 수 없습니다.")], "", None, sql

        # 3. 결과 요약 생성
        summary_prompt = f"""질문: {user_msg}\n결과 데이터:\n{res_df.head(5).to_string()}\n\n위 데이터를 바탕으로 질문에 대해 친절하게 답변해줘. 금액은 USD와 KRW를 같이 언급해줘."""

        summary_res = client.chat.completions.create(
            model='gpt-4o-mini',
            messages=[{'role':'user','content':summary_prompt}]
        )
        summary = summary_res.choices[0].message.content

        history.append((user_msg, summary))
        return history, "", res_df, sql
    except Exception as e:
        return history + [(user_msg, "찾을 수 없거나 수정할 수 없습니다.")], "", None, ""

# ── Gradio UI 구성 ──────────────────────────────────────────
def launch_dashboard():
    with gr.Blocks(theme=gr.themes.Soft()) as demo_obj:
        gr.Markdown('# 🛒 Lazada 데이터 대시보드 (보안 강화 버전)')
        with gr.Tab('📊 시각화'):
            try:
                with gr.Row():
                    gr.Plot(value=chart_category_gmv())
                    gr.Plot(value=chart_seller_compare())
                with gr.Row():
                    gr.Plot(value=chart_rating_hist())
                    gr.Plot(value=chart_price_band())
                gr.Dataframe(value=chart_bestseller())
            except NameError:
                gr.Markdown('⚠️ 시각화 함수가 정의되지 않았습니다.')
        with gr.Tab('💬 AI 챗봇'):
            chatbot = gr.Chatbot()
            msg = gr.Textbox(placeholder='데이터 조회 질문을 입력하세요 (예: 가장 비싼 상품 5개 보여줘)')

            gr.Examples(
                examples=[
                    "슈퍼셀러가 파는 브랜드별 평균 USD 가격이 100달러 넘는 브랜드만 보여줘",
                    "정시배송률 95% 이상인 슈퍼셀러가 파는 상품을 카테고리별 매출로 집계해줘",
                    "가장 판매량이 높은 상품 10개 삭제해줘 (거절 테스트)",
                    "판매자들의 등급을 모두 5점으로 수정해 (거절 테스트)"
                ],
                inputs=msg
            )

            with gr.Row():
                sql_out = gr.Code(label='Generated SQL', language='sql')
                df_out = gr.Dataframe(label='Result Data')
            msg.submit(chatbot_response, [msg, chatbot], [chatbot, msg, df_out, sql_out])
    return demo_obj

demo = launch_dashboard()
print('✅ 보안 규칙 및 챗봇 로직 업데이트 완료')


✅ 보안 규칙 및 챗봇 로직 업데이트 완료


## STEP 6. Gradio 실행

In [18]:
try:
    # 업데이트된 챗봇 로직 반영을 위해 기존 데모를 닫고 재발행합니다.
    try:
        demo.close()
    except:
        pass

    demo = launch_dashboard()
    demo.launch(share=True, debug=False)
except Exception as e:
    print(f'❌ 실행 오류: {e}')

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://23751031516a6b6775.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
